# 06 — Đánh giá Chất lượng Dự đoán và Thiên lệch Độ phổ biến ở Đầu ra trên Yoochoose

Notebook này tổng hợp và phân tích toàn diện đầu ra **Top-20** của **3 mô hình SBR (`GRU4Rec`, `NARM`, `SASRec`) × 2 random seed (`2024`, `2025`)** trên bộ dữ liệu **Yoochoose `1/32`** để trả lời **Câu hỏi nghiên cứu 1**:

> **Câu hỏi 1:** *Mô hình nào có thiên lệch độ phổ biến (Popularity Bias) lớn nhất theo `Gini@20`, `Coverage@20`, `ARP@20` và tỷ trọng exposure `Popular/Rare` trên Yoochoose? Trade-off giữa Accuracy và Diversity của ba mô hình ra sao, và hiện tượng này có lặp lại so với Diginetica không?*

### Cấu trúc phân tích chi tiết (10 phần)
1. Kiểm tra tính đầy đủ và toàn vẹn của 6 lần chạy (`runs/yoochoose/` & `results/yoochoose/item_level/`).
2. Tổng hợp Accuracy tổng thể (`All-prefix`, `Last-item`, đối chứng `Full-test` có Cold-start 20% và thời gian train).
3. Phân tầng Accuracy theo nhóm target (`Popular` / `Medium` / `Rare`) & đo khoảng cách `Popular − Rare`.
4. Kiểm toán lại độc lập các chỉ số Popularity Bias từ `top20.parquet` (`Coverage@20`, `Gini@20`, `ARP@20`).
5. Phân bổ Exposure Top-20, `Exposure-to-Train Ratio` và `Within-group Coverage` (bao gồm số item không bao giờ được gợi ý).
6. Mức độ tập trung: Đường cong Lorenz, Top-1%/5%/10% item concentration và **Hồ sơ độ phổ biến theo thứ hạng Rank 1 → Rank 20**.
7. **Phân tích đặc thù Yoochoose**: Mức độ khuếch đại exposure của nhóm **Special Offer (`'S'`)** so với nhóm không gắn nhãn `'S'`.
8. Trực quan hóa `Coverage@20`, `ARP@20` và đánh đổi `Accuracy ↔ Diversity`.
9. **So sánh chéo trực tiếp Diginetica vs. Yoochoose** trên cùng 3 kiến trúc mô hình.
10. Tổng hợp kết luận File 06 & bàn giao sang File 07 (Hình học không gian Embedding).

## 1. Thiết lập môi trường, đường dẫn và kiểm tra đủ 6 lần chạy Yoochoose

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án DA-3.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from btl_k03.paths import get_dataset_paths, ensure_output_dirs
from btl_k03.utils import load_item_mapping, load_json
from btl_k03.popularity_metrics import (
    compute_gini_coefficient,
    compute_lorenz_curve,
    evaluate_top20_popularity_bias,
)

DATASET = "yoochoose"
MODELS = ["gru4rec", "narm", "sasrec"]
MODEL_LABELS = {"gru4rec": "GRU4Rec", "narm": "NARM", "sasrec": "SASRec"}
MODEL_COLORS = {"GRU4REC": "#3182ce", "NARM": "#dd6b20", "SASREC": "#2f855a"}
SEEDS = [2024, 2025]
TOPK = 20

ensure_output_dirs(DATASET)
YOO_PATHS = get_dataset_paths(DATASET)
DIGI_PATHS = get_dataset_paths("diginetica")

PROCESSED_DIR = YOO_PATHS["processed_dir"]
RUNS_DIR = YOO_PATHS["runs_dir"]
TABLES_DIR = YOO_PATHS["tables_dir"]
FIGURES_DIR = YOO_PATHS["figures_dir"]
ITEM_LEVEL_DIR = YOO_PATHS["item_level_dir"]

item_mapping = load_item_mapping(PROCESSED_DIR)
dataset_stats = load_json(PROCESSED_DIR / "dataset_statistics.json")

# Kiểm tra đủ 6 lần chạy
check_rows = []
for m in MODELS:
    for seed in SEEDS:
        rdir = RUNS_DIR / m / f"seed_{seed}"
        tm_ok = (rdir / "test_metrics.json").is_file()
        top_ok = (rdir / "top20.parquet").is_file()
        il_ok = (ITEM_LEVEL_DIR / f"{m}_seed{seed}.parquet").is_file()
        check_rows.append(
            {
                "model": m.upper(),
                "seed": seed,
                "test_metrics_ok": tm_ok,
                "top20_ok": top_ok,
                "item_level_ok": il_ok,
                "status": "OK" if (tm_ok and top_ok and il_ok) else "MISSING",
            }
        )

check_df = pd.DataFrame(check_rows)
assert (check_df["status"] == "OK").all(), f"Thiếu kết quả huấn luyện Yoochoose!\n{check_df}"
display(check_df)

n_warm = int(item_mapping["is_trainable_warm"].sum())
n_cold = int(item_mapping["is_controlled_cold"].sum())
test_queries = int(next(s["prefix_target_examples"] for s in dataset_stats["final_splits"] if s["stage"] == "test_warm"))
print(
    f"Quy mô đánh giá Yoochoose: {test_queries:,} warm test queries × Top-{TOPK} = "
    f"{test_queries * TOPK:,} slots/lần chạy | Warm catalog = {n_warm:,} | Controlled cold = {n_cold:,}"
)

## 2. Tổng hợp chất lượng dự đoán tổng thể (`All-prefix`, `Last-item`, `Cold-start Diagnostic` & `Runtime`)

Báo cáo đồng thời độ chính xác trên tập Warm chính thức, Last-item, đối chứng Full-test (coi target cold là 0) và thời gian huấn luyện để đảm bảo các mô hình đều hội tụ chuẩn trước khi kết luận về thiên lệch độ phổ biến.

In [ ]:
acc_rows = []
cold_diag_rows = []
strat_rows = []
loaded_runs: dict[tuple[str, int], dict] = {}

for m in MODELS:
    for seed in SEEDS:
        rdir = RUNS_DIR / m / f"seed_{seed}"
        tm = load_json(rdir / "test_metrics.json")
        top20 = pd.read_parquet(rdir / "top20.parquet")
        il_df = pd.read_parquet(ITEM_LEVEL_DIR / f"{m}_seed{seed}.parquet")
        loaded_runs[(m, seed)] = {"tm": tm, "top20": top20, "il": il_df}

        w_all = tm["test_warm_all_prefix"]
        w_last = tm["test_warm_last_item"]
        cd = tm["cold_start_diagnostic"]
        rt = tm.get("runtime", {})

        acc_rows.append(
            {
                "model": m.upper(),
                "seed": seed,
                "epochs_ran": rt.get("epochs_ran", tm["best_epoch"]),
                "best_epoch": tm["best_epoch"],
                "train_time_min": rt.get("total_train_time_min", np.nan),
                "val_best_mrr@20": tm["best_validation_score"],
                "recall@20": w_all["recall@20"],
                "mrr@20": w_all["mrr@20"],
                "ndcg@20": w_all["ndcg@20"],
                "last_item_recall@20": w_last["recall@20"],
                "last_item_mrr@20": w_last["mrr@20"],
                "last_item_ndcg@20": w_last["ndcg@20"],
                "full_test_effective_recall@20": cd["effective_full_test_recall@20_with_cold_zero"],
                "cold_max_drift": tm["cold_embedding_max_abs_drift"],
            }
        )
        cold_diag_rows.append({"model": m.upper(), "seed": seed, **cd})
        for rec in tm["test_warm_by_popularity_group"]:
            strat_rows.append({"model": m.upper(), "seed": seed, **rec})

model_accuracy_df = pd.DataFrame(acc_rows)
model_accuracy_df.to_csv(TABLES_DIR / "model_accuracy.csv", index=False)

cold_diag_df = pd.DataFrame(cold_diag_rows)
cold_diag_df.to_csv(TABLES_DIR / "cold_start_diagnostic.csv", index=False)

acc_metrics = [
    "recall@20", "mrr@20", "ndcg@20",
    "last_item_recall@20", "last_item_mrr@20",
    "full_test_effective_recall@20", "train_time_min",
]
acc_seed_summary = (
    model_accuracy_df.groupby("model", sort=False)[acc_metrics]
    .agg(["mean", "std"])
)
acc_seed_summary.columns = [f"{col}_{stat}" for col, stat in acc_seed_summary.columns]
acc_seed_summary = acc_seed_summary.reset_index()
acc_seed_summary.to_csv(TABLES_DIR / "model_accuracy_seed_summary.csv", index=False)

print("=== BẢNG 1A: ACCURACY CHI TIẾT 6 LẦN CHẠY (YOOCHOOSE) ===")
display(model_accuracy_df)

print("\n=== BẢNG 1B: ACCURACY TRUNG BÌNH ± ĐỘ LỆCH CHUẨN QUA 2 SEED ===")
display(acc_seed_summary)

print("\n=== BẢNG 1C: CHẨN ĐOÁN COLD-START TRÊN FULL TEST (SEED 2024 ĐẠI DIỆN) ===")
display(
    cold_diag_df.loc[cold_diag_df["seed"] == 2024, [
        "model", "full_test_all_prefix_targets", "warm_evaluated_targets",
        "controlled_cold_targets_in_test", "controlled_cold_target_share_pct",
        "natural_cold_targets_in_test", "natural_cold_target_share_pct",
        "warm_recall@20", "effective_full_test_recall@20_with_cold_zero",
    ]]
)

## 3. Phân tầng Accuracy theo nhóm độ phổ biến của Target (`Popular` / `Medium` / `Rare`)

Kiểm tra xem trên Yoochoose, sự bất bình đẳng có xuất hiện ngay trong **khả năng dự đoán đúng** hay không (chênh lệch `Recall@20` và `MRR@20` giữa target `Popular` và target `Rare`).

In [ ]:
strat_df = pd.DataFrame(strat_rows)
strat_df.to_csv(TABLES_DIR / "stratified_accuracy.csv", index=False)

strat_summary = (
    strat_df.groupby(["model", "target_group"], sort=False)[["num_examples", "example_share_pct", "recall@20", "mrr@20", "ndcg@20"]]
    .mean()
    .reset_index()
)
strat_summary.to_csv(TABLES_DIR / "stratified_accuracy_seed_summary.csv", index=False)

# Tính khoảng cách Popular - Rare cho từng mô hình
gap_rows = []
for m_up in ["GRU4REC", "NARM", "SASREC"]:
    sub = strat_summary.loc[strat_summary["model"] == m_up].set_index("target_group")
    gap_rows.append(
        {
            "model": m_up,
            "popular_recall@20": sub.loc["popular", "recall@20"],
            "medium_recall@20": sub.loc["medium", "recall@20"],
            "rare_recall@20": sub.loc["rare", "recall@20"],
            "recall_gap_pop_minus_rare_pp": (sub.loc["popular", "recall@20"] - sub.loc["rare", "recall@20"]) * 100.0,
            "popular_mrr@20": sub.loc["popular", "mrr@20"],
            "rare_mrr@20": sub.loc["rare", "mrr@20"],
            "mrr_gap_pop_minus_rare_pp": (sub.loc["popular", "mrr@20"] - sub.loc["rare", "mrr@20"]) * 100.0,
        }
    )
strat_gap_df = pd.DataFrame(gap_rows)
strat_gap_df.to_csv(TABLES_DIR / "stratified_accuracy_gaps.csv", index=False)

print("=== BẢNG 2A: ACCURACY PHÂN TẦNG TRUNG BÌNH QUA 2 SEED ===")
display(strat_summary)

print("\n=== BẢNG 2B: CHÊNH LỆCH ACCURACY GIỮA POPULAR VÀ RARE TARGET (ĐIỂM %) ===")
display(strat_gap_df)

plot_strat = strat_summary.loc[strat_summary["target_group"].isin(["popular", "medium", "rare"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
sns.barplot(
    data=plot_strat, x="target_group", y="recall@20", hue="model",
    order=["popular", "medium", "rare"], hue_order=["GRU4REC", "NARM", "SASREC"],
    palette=MODEL_COLORS, ax=axes[0],
)
axes[0].set_title("Recall@20 theo nhóm độ phổ biến của Target (Yoochoose)")
axes[0].set_xlabel("Nhóm Target")
axes[0].set_ylabel("Recall@20")
axes[0].set_ylim(0, 0.90)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.3f", fontsize=8, padding=2)

sns.barplot(
    data=plot_strat, x="target_group", y="mrr@20", hue="model",
    order=["popular", "medium", "rare"], hue_order=["GRU4REC", "NARM", "SASREC"],
    palette=MODEL_COLORS, ax=axes[1],
)
axes[1].set_title("MRR@20 theo nhóm độ phổ biến của Target (Yoochoose)")
axes[1].set_xlabel("Nhóm Target")
axes[1].set_ylabel("MRR@20")
axes[1].set_ylim(0, 0.42)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.3f", fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stratified_accuracy_by_group.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Kiểm toán độc lập các chỉ số Popularity Bias từ `top20.parquet` (`Coverage@20`, `Gini@20`, `ARP@20`)

Tính lại trực tiếp toàn bộ chỉ số từ `top20.parquet` và `item_mapping.csv` qua `evaluate_top20_popularity_bias()` để kiểm chứng chéo với `test_metrics.json`.

In [ ]:
bias_rows = []
exposure_frames = []
catalog_rec_tables: dict[tuple[str, int], pd.DataFrame] = {}

for m in MODELS:
    for seed in SEEDS:
        top20 = loaded_runs[(m, seed)]["top20"]
        tm_pb = loaded_runs[(m, seed)]["tm"]["popularity_bias"]

        pb, cat_df, grp_exp_df = evaluate_top20_popularity_bias(top20, item_mapping, k=TOPK)
        # Kiểm chứng khớp tuyệt đối với test_metrics.json
        assert np.isclose(pb["gini_warm@20"], tm_pb["gini_warm@20"], atol=1e-9)
        assert np.isclose(pb["coverage_warm@20"], tm_pb["coverage_warm@20"], atol=1e-9)
        assert np.isclose(pb["arp@20"], tm_pb["arp@20"], atol=1e-6)

        catalog_rec_tables[(m, seed)] = cat_df
        bias_rows.append({"model": m.upper(), "seed": seed, **pb})

        grp_copy = grp_exp_df.copy()
        grp_copy.insert(0, "seed", seed)
        grp_copy.insert(0, "model", m.upper())
        exposure_frames.append(grp_copy)

popularity_bias_df = pd.DataFrame(bias_rows)
popularity_bias_df.to_csv(TABLES_DIR / "popularity_bias.csv", index=False)

exposure_by_group_df = pd.concat(exposure_frames, ignore_index=True)
exposure_by_group_df.to_csv(TABLES_DIR / "exposure_by_group.csv", index=False)

pb_cols = [
    "coverage_warm@20", "coverage_full@20",
    "gini_warm@20", "gini_full@20",
    "arp@20", "arp_log1p@20",
    "popular_group_exposure_share_pct", "rare_group_exposure_share_pct",
]
pb_seed_summary = popularity_bias_df.groupby("model", sort=False)[pb_cols].agg(["mean", "std"])
pb_seed_summary.columns = [f"{c}_{s}" for c, s in pb_seed_summary.columns]
pb_seed_summary = pb_seed_summary.reset_index()
pb_seed_summary.to_csv(TABLES_DIR / "popularity_bias_seed_summary.csv", index=False)

print("=== BẢNG 3A: CHỈ SỐ POPULARITY BIAS ĐẦU RA (6 LẦN CHẠY — ĐÃ KIỂM CHỨNG TỪ TOP20.PARQUET) ===")
display(popularity_bias_df[["model", "seed", "unique_recommended_items_warm", *pb_cols]])

print("\n=== BẢNG 3B: TỔNG HỢP TRUNG BÌNH ± STD QUA 2 SEED ===")
display(pb_seed_summary)

## 5. Phân bổ Exposure Top-20, `Exposure-to-Train Ratio` và `Within-Group Coverage`

Đánh giá xem các mô hình chỉ **phản ánh lại** độ lệch của tập Train hay đang **khuếch đại** độ lệch đó (`Exposure-to-Train Ratio > 1` với nhóm Popular và `< 1` với nhóm Rare), đồng thời đếm số item bị "vô hình" (`rec_count == 0`).

In [ ]:
exp_seed_summary = (
    exposure_by_group_df.groupby(["model", "popularity_group"], sort=False)
    .agg(
        catalog_items=("catalog_items", "first"),
        warm_catalog_share_pct=("warm_catalog_share_pct", "first"),
        train_interaction_share_pct=("train_interaction_share_pct", "first"),
        exposure_share_pct_mean=("exposure_share_pct", "mean"),
        exposure_share_pct_std=("exposure_share_pct", "std"),
        exposure_to_train_ratio_mean=("exposure_to_train_ratio", "mean"),
        within_group_coverage_pct_mean=("within_group_coverage_pct", "mean"),
        mean_rec_count_per_item=("mean_rec_count_per_item", "mean"),
        median_rec_count_per_item=("median_rec_count_per_item", "mean"),
    )
    .reset_index()
)
exp_seed_summary["never_recommended_items_est"] = (
    exp_seed_summary["catalog_items"] * (1.0 - exp_seed_summary["within_group_coverage_pct_mean"] / 100.0)
).round(1)
exp_seed_summary.to_csv(TABLES_DIR / "exposure_by_group_seed_summary.csv", index=False)

print("=== BẢNG 4: PHÂN BỔ EXPOSURE, TỶ SỐ KHUẾCH ĐẠI & COVERAGE THEO NHÓM (TB 2 SEED) ===")
display(exp_seed_summary)

warm_exp = exp_seed_summary.loc[exp_seed_summary["popularity_group"].isin(["popular", "medium", "rare"])].copy()
groups_order = ["popular", "medium", "rare"]
ref_group = warm_exp.loc[warm_exp["model"] == "GRU4REC"].set_index("popularity_group").loc[groups_order]

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8))
x = np.arange(len(groups_order))
w = 0.16
axes[0].bar(x - 2 * w, ref_group["warm_catalog_share_pct"], w, label="% Warm Catalog", color="#a0aec0")
axes[0].bar(x - 1 * w, ref_group["train_interaction_share_pct"], w, label="% Train Clicks", color="#4a5568")
for offset, m_up in [(0, "GRU4REC"), (1, "NARM"), (2, "SASREC")]:
    vals = warm_exp.loc[warm_exp["model"] == m_up].set_index("popularity_group").loc[groups_order, "exposure_share_pct_mean"]
    axes[0].bar(x + offset * w, vals, w, label=f"{m_up} Exposure", color=MODEL_COLORS[m_up])

axes[0].set_xticks(x)
axes[0].set_xticklabels(["Popular", "Medium", "Rare"])
axes[0].set_ylabel("Tỷ trọng (%)")
axes[0].set_title("Catalog vs. Train Clicks vs. Top-20 Exposure (Yoochoose)")
axes[0].legend(fontsize=8.5, ncol=2)

sns.barplot(
    data=warm_exp, x="popularity_group", y="exposure_to_train_ratio_mean", hue="model",
    order=groups_order, hue_order=["GRU4REC", "NARM", "SASREC"],
    palette=MODEL_COLORS, ax=axes[1],
)
axes[1].axhline(1.0, color="#e53e3e", linestyle="--", linewidth=1.5, label="Tỷ lệ công bằng theo Train (= 1.0)")
axes[1].set_title("Hệ số khuếch đại Exposure-to-Train Ratio (>1: khuếch đại, <1: suy giảm)")
axes[1].set_xlabel("Nhóm độ phổ biến")
axes[1].set_ylabel("Exposure % / Train Click %")
axes[1].set_ylim(0, 1.25)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.2f", fontsize=8.5, padding=2)
axes[1].legend(fontsize=8.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "exposure_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Mức độ tập trung: Đường cong Lorenz, Top-1%/5%/10% Item Concentration & Hồ sơ độ phổ biến theo thứ hạng (`Rank 1 → Rank 20`)

Phân tích sâu hai khía cạnh quan trọng của đầu ra Top-20:
1. **Đường cong Lorenz & Top-K% item concentration**: Top 1% / 5% / 10% warm item được gợi ý nhiều nhất chiếm bao nhiêu phần trăm tổng `831.700` vị trí Top-20? So sánh trực tiếp với đường cong Lorenz của tập Train đầu vào.
2. **Độ phổ biến theo vị trí xếp hạng (`Rank 1 → Rank 20`)**: Mô hình có đẩy item phổ biến lên các vị trí đầu danh sách (`Rank 1–5`) mạnh hơn cuối danh sách (`Rank 16–20`) không?

In [ ]:
# 1. Tính Top-1/5/10/20% recommendation concentration + Train baseline
warm_train_freqs = item_mapping.loc[item_mapping["is_trainable_warm"], "train_frequency"].to_numpy(dtype=np.float64)
train_gini = compute_gini_coefficient(warm_train_freqs)
train_lx, train_ly = compute_lorenz_curve(warm_train_freqs)

conc_rows = []
for pct in (1, 5, 10, 20):
    k_items = max(1, int(round(len(warm_train_freqs) * pct / 100.0)))
    s_tr = np.sort(warm_train_freqs)[::-1]
    conc_rows.append(
        {"source": "TRAIN_INPUT", "seed": 0, "top_item_pct": pct, "num_items": k_items, "share_pct": float(s_tr[:k_items].sum() / s_tr.sum() * 100.0)}
    )
    for m in MODELS:
        for seed in SEEDS:
            cat_w = catalog_rec_tables[(m, seed)]
            rc = cat_w.loc[cat_w["candidate_for_main_evaluation"], "rec_count"].to_numpy(dtype=np.float64)
            s_rc = np.sort(rc)[::-1]
            conc_rows.append(
                {"source": m.upper(), "seed": seed, "top_item_pct": pct, "num_items": k_items, "share_pct": float(s_rc[:k_items].sum() / s_rc.sum() * 100.0)}
            )

conc_df = pd.DataFrame(conc_rows)
conc_summary = conc_df.groupby(["source", "top_item_pct"], sort=False)["share_pct"].mean().unstack("top_item_pct").reset_index()
conc_summary.columns = ["source", "top_1pct_share", "top_5pct_share", "top_10pct_share", "top_20pct_share"]
conc_summary.to_csv(TABLES_DIR / "recommendation_concentration.csv", index=False)
print(f"Train Warm Input Gini = {train_gini:.4f}")
display(conc_summary)

# 2. Hồ sơ độ phổ biến theo thứ hạng Rank 1..20
freq_map = item_mapping.set_index("item_id")["train_frequency"]
group_map = item_mapping.set_index("item_id")["popularity_group"]

rank_rows = []
for m in MODELS:
    for seed in SEEDS:
        t20 = loaded_runs[(m, seed)]["top20"][["rank", "item_id"]].copy()
        t20["train_freq"] = t20["item_id"].map(freq_map).to_numpy(dtype=np.float64)
        t20["is_pop"] = (t20["item_id"].map(group_map) == "popular").to_numpy(dtype=np.float64)
        t20["is_rare"] = (t20["item_id"].map(group_map) == "rare").to_numpy(dtype=np.float64)
        by_rank = t20.groupby("rank", sort=True).agg(
            mean_arp=("train_freq", "mean"),
            popular_share_pct=("is_pop", lambda s: float(s.mean() * 100.0)),
            rare_share_pct=("is_rare", lambda s: float(s.mean() * 100.0)),
        ).reset_index()
        by_rank.insert(0, "seed", seed)
        by_rank.insert(0, "model", m.upper())
        rank_rows.append(by_rank)

rank_profile_df = pd.concat(rank_rows, ignore_index=True)
rank_profile_mean = rank_profile_df.groupby(["model", "rank"], sort=False)[["mean_arp", "popular_share_pct", "rare_share_pct"]].mean().reset_index()
rank_profile_mean.to_csv(TABLES_DIR / "rank_popularity_profile.csv", index=False)

# Biểu đồ 1: Gini + Lorenz Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
gini_means = pb_seed_summary.set_index("model").loc[["GRU4REC", "NARM", "SASREC"], "gini_warm@20_mean"]
gini_stds = pb_seed_summary.set_index("model").loc[["GRU4REC", "NARM", "SASREC"], "gini_warm@20_std"]
bars = axes[0].bar(
    ["GRU4Rec", "NARM", "SASRec"], gini_means.values, yerr=gini_stds.values,
    capsize=5, color=[MODEL_COLORS[k] for k in ["GRU4REC", "NARM", "SASREC"]], alpha=0.9,
)
axes[0].axhline(train_gini, color="#e53e3e", linestyle="--", linewidth=1.6, label=f"Train Input Gini ({train_gini:.4f})")
axes[0].set_ylim(0.75, 0.94)
axes[0].set_ylabel("Gini@20 (Warm Catalog)")
axes[0].set_title("So sánh Recommendation Gini@20 với Train Input Gini")
axes[0].legend(loc="lower right")
for bar, val in zip(bars, gini_means.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.004, f"{val:.4f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold")

axes[1].plot([0, 100], [0, 100], linestyle=":", color="#718096", label="Phân bổ đều hoàn toàn")
axes[1].plot(train_lx * 100, train_ly * 100, linestyle="--", color="#1a202c", linewidth=2, label=f"Train Input (Gini={train_gini:.3f})")
for m in MODELS:
    rc24 = catalog_rec_tables[(m, 2024)].loc[lambda d: d["candidate_for_main_evaluation"], "rec_count"].to_numpy()
    lx, ly = compute_lorenz_curve(rc24)
    gval = pb_seed_summary.set_index("model").loc[m.upper(), "gini_warm@20_mean"]
    axes[1].plot(lx * 100, ly * 100, linewidth=2.2, color=MODEL_COLORS[m.upper()], label=f"{MODEL_LABELS[m]} (Gini={gval:.3f})")
axes[1].set_title("Đường cong Lorenz của Top-20 Exposure (Yoochoose)")
axes[1].set_xlabel("Tỷ lệ tích lũy Warm Catalog (%)")
axes[1].set_ylabel("Tỷ lệ tích lũy Exposure (%)")
axes[1].legend(loc="upper left", fontsize=8.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "gini_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Biểu đồ 2: Độ phổ biến theo thứ hạng Rank 1..20
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
for m_up in ["GRU4REC", "NARM", "SASREC"]:
    sub = rank_profile_mean.loc[rank_profile_mean["model"] == m_up]
    axes[0].plot(sub["rank"], sub["mean_arp"], marker="o", markersize=4, linewidth=2, color=MODEL_COLORS[m_up], label=m_up)
    axes[1].plot(sub["rank"], sub["popular_share_pct"], marker="s", markersize=4, linewidth=2, color=MODEL_COLORS[m_up], label=m_up)

axes[0].set_title("ARP trung bình theo thứ hạng trong danh sách Top-20 (Rank 1 → 20)")
axes[0].set_xlabel("Thứ hạng gợi ý (Rank)")
axes[0].set_ylabel("Mean Train Frequency (ARP)")
axes[0].set_xticks(range(1, 21, 2))
axes[0].legend()

axes[1].set_title("Tỷ trọng Popular Item theo thứ hạng (Rank 1 → 20)")
axes[1].set_xlabel("Thứ hạng gợi ý (Rank)")
axes[1].set_ylabel("Popular Share (%)")
axes[1].set_xticks(range(1, 21, 2))
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "rank_popularity_profile.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Phân tích đặc thù Yoochoose: Hiệu ứng khuếch đại nhóm `Special Offer ('S')`

Ở bước EDA (`01_eda.ipynb`), ta phát hiện nhóm sản phẩm từng gắn nhãn `'S'` chỉ chiếm một tỷ lệ nhỏ catalog nhưng tạo ra hơn nửa lượt click.
Cell này kiểm tra xem **GRU4Rec, NARM và SASRec có tiếp tục khuếch đại nhóm `'S'` ở đầu ra Top-20 vượt quá cả tỷ trọng click trong Train hay không**.

In [ ]:
so_rows = []
for m in MODELS:
    for seed in SEEDS:
        il = loaded_runs[(m, seed)]["il"]
        warm_il = il.loc[il["is_trainable_warm"]].copy()
        total_slots = max(1, int(warm_il["rec_count"].sum()))
        total_train = max(1, int(warm_il["train_freq"].sum()))
        for flag, lbl in [(True, "Special Offer ('S')"), (False, "Non-Special Offer")]:
            sub = warm_il.loc[warm_il["ever_special_offer"] == flag]
            train_share = sub["train_freq"].sum() / total_train * 100.0
            exp_share = sub["rec_count"].sum() / total_slots * 100.0
            so_rows.append(
                {
                    "model": m.upper(),
                    "seed": seed,
                    "special_offer_group": lbl,
                    "warm_items": len(sub),
                    "warm_catalog_pct": len(sub) / len(warm_il) * 100.0,
                    "train_click_share_pct": train_share,
                    "exposure_share_pct": exp_share,
                    "exposure_to_train_ratio": exp_share / max(train_share, 1e-9),
                    "within_group_coverage_pct": float((sub["rec_count"] > 0).mean() * 100.0),
                    "mean_rec_count": float(sub["rec_count"].mean()),
                    "median_rec_count": float(sub["rec_count"].median()),
                }
            )

so_df = pd.DataFrame(so_rows)
so_summary = (
    so_df.groupby(["model", "special_offer_group"], sort=False)
    .agg(
        warm_items=("warm_items", "first"),
        warm_catalog_pct=("warm_catalog_pct", "first"),
        train_click_share_pct=("train_click_share_pct", "first"),
        exposure_share_pct_mean=("exposure_share_pct", "mean"),
        exposure_to_train_ratio_mean=("exposure_to_train_ratio", "mean"),
        within_group_coverage_pct=("within_group_coverage_pct", "mean"),
        mean_rec_count=("mean_rec_count", "mean"),
    )
    .reset_index()
)
so_df.to_csv(TABLES_DIR / "special_offer_exposure_by_seed.csv", index=False)
so_summary.to_csv(TABLES_DIR / "special_offer_exposure.csv", index=False)
display(so_summary)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
so_only = so_summary.loc[so_summary["special_offer_group"] == "Special Offer ('S')"].set_index("model").loc[["GRU4REC", "NARM", "SASREC"]]
cat_pct = float(so_only["warm_catalog_pct"].iloc[0])
tr_pct = float(so_only["train_click_share_pct"].iloc[0])

bar_labels = ["% Warm Catalog", "% Train Clicks", "GRU4Rec Exp", "NARM Exp", "SASRec Exp"]
bar_vals = [cat_pct, tr_pct, *so_only["exposure_share_pct_mean"].tolist()]
bar_cols = ["#a0aec0", "#4a5568", MODEL_COLORS["GRU4REC"], MODEL_COLORS["NARM"], MODEL_COLORS["SASREC"]]
bars = axes[0].bar(bar_labels, bar_vals, color=bar_cols, alpha=0.9)
axes[0].set_ylabel("Tỷ trọng (%)")
axes[0].set_title("Nhóm Special Offer ('S'): Catalog vs. Train Clicks vs. Top-20 Exposure")
axes[0].tick_params(axis="x", rotation=15)
for b, v in zip(bars, bar_vals):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 1.0, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)

sns.barplot(
    data=so_summary, x="special_offer_group", y="exposure_to_train_ratio_mean", hue="model",
    hue_order=["GRU4REC", "NARM", "SASREC"], palette=MODEL_COLORS, ax=axes[1],
)
axes[1].axhline(1.0, color="#e53e3e", linestyle="--", linewidth=1.5)
axes[1].set_title("Hệ số khuếch đại Exposure-to-Train: Nhóm 'S' vs. Không 'S'")
axes[1].set_xlabel("")
axes[1].set_ylabel("Exposure % / Train Click %")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.3f", fontsize=8.5, padding=2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "special_offer_bias_amplification.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Trực quan hóa `Catalog Coverage@20`, `ARP@20` và đánh đổi `Accuracy ↔ Diversity`

In [ ]:
# Biểu đồ 1: Catalog Coverage@20 (Warm, Full và Within-group Rare)
cov_order = ["GRU4REC", "NARM", "SASREC"]
pb_idx = pb_seed_summary.set_index("model").loc[cov_order]
rare_cov = (
    exp_seed_summary.loc[exp_seed_summary["popularity_group"] == "rare"]
    .set_index("model")
    .loc[cov_order, "within_group_coverage_pct_mean"]
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
x = np.arange(3)
w = 0.28
b1 = axes[0].bar(x - w / 2, pb_idx["coverage_warm@20_mean"] * 100, w, yerr=pb_idx["coverage_warm@20_std"] * 100, capsize=4, label="Warm Coverage@20 (%)", color="#3182ce")
b2 = axes[0].bar(x + w / 2, rare_cov.values, w, label="Rare Within-Group Coverage (%)", color="#ed8936")
axes[0].set_xticks(x)
axes[0].set_xticklabels(["GRU4Rec", "NARM", "SASRec"])
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Coverage (%)")
axes[0].set_title("Catalog Coverage@20 tổng thể & trong nhóm Rare (Yoochoose)")
axes[0].legend()
for b in list(b1) + list(b2):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 1.2, f"{b.get_height():.1f}%", ha="center", va="bottom", fontsize=8.5)

# Biểu đồ 2: Trade-off Recall@20 vs. Gini_warm@20
acc_idx = acc_seed_summary.set_index("model").loc[cov_order]
for m_up in cov_order:
    sub_pts = model_accuracy_df.merge(popularity_bias_df, on=["model", "seed"]).loc[lambda d: d["model"] == m_up]
    axes[1].scatter(
        sub_pts["gini_warm@20"], sub_pts["recall@20"],
        s=110, color=MODEL_COLORS[m_up], label=m_up, alpha=0.9, edgecolors="black", linewidth=0.7,
    )
    for _, row in sub_pts.iterrows():
        axes[1].annotate(f"{m_up[:3]}-{int(row['seed'])%100}", (row["gini_warm@20"], row["recall@20"]), xytext=(5, 4), textcoords="offset points", fontsize=8.5)

axes[1].invert_xaxis()  # Bên phải là Gini thấp hơn (tốt hơn)
axes[1].set_xlabel("Gini_warm@20 (trục đảo ngược: sang phải = ít thiên lệch hơn)")
axes[1].set_ylabel("Recall@20 (cao hơn = chính xác hơn)")
axes[1].set_title("Đánh đổi Accuracy ↔ Diversity (Góc trên bên phải tối ưu nhất)")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "catalog_coverage.png", dpi=160, bbox_inches="tight")
plt.show()

# Biểu đồ ARP@20 (Raw và Log1p)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
b_raw = axes[0].bar(
    ["GRU4Rec", "NARM", "SASRec"], pb_idx["arp@20_mean"], yerr=pb_idx["arp@20_std"],
    capsize=5, color=[MODEL_COLORS[k] for k in cov_order], alpha=0.9,
)
axes[0].set_title("Average Recommendation Popularity — ARP@20 (thấp hơn = ít thiên lệch hơn)")
axes[0].set_ylabel("ARP@20 (Mean Train Frequency)")
axes[0].set_ylim(600, 830)
for b, v in zip(b_raw, pb_idx["arp@20_mean"]):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 5, f"{v:.1f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold")

b_log = axes[1].bar(
    ["GRU4Rec", "NARM", "SASRec"], pb_idx["arp_log1p@20_mean"], yerr=pb_idx["arp_log1p@20_std"],
    capsize=5, color=[MODEL_COLORS[k] for k in cov_order], alpha=0.9,
)
axes[1].set_title("ARP_log1p@20 (thang đo log(1 + train_freq))")
axes[1].set_ylabel("Mean log(1 + train_freq)")
axes[1].set_ylim(5.4, 5.95)
for b, v in zip(b_log, pb_idx["arp_log1p@20_mean"]):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.3f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "average_popularity.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Đối chiếu chéo trực tiếp: **Diginetica vs. Yoochoose** (Accuracy & Popularity Bias)

Đọc trực tiếp các bảng kết quả Diginetica tại `results/tables/` để đặt song song với Yoochoose trên cùng 3 kiến trúc `GRU4Rec`, `NARM`, `SASRec`.

In [ ]:
digi_acc = pd.read_csv(DIGI_PATHS["tables_dir"] / "model_accuracy.csv")
digi_pb = pd.read_csv(DIGI_PATHS["tables_dir"] / "popularity_bias.csv")


def summarize_dataset(acc_frame: pd.DataFrame, pb_frame: pd.DataFrame, dataset_label: str) -> pd.DataFrame:
    merged = acc_frame.merge(pb_frame, on=["model", "seed"])
    out = (
        merged.groupby("model", sort=False)
        .agg(
            recall20=("recall@20", "mean"),
            mrr20=("mrr@20", "mean"),
            ndcg20=("ndcg@20", "mean"),
            coverage_warm_pct=("coverage_warm@20", lambda s: float(s.mean() * 100.0)),
            gini_warm=("gini_warm@20", "mean"),
            arp20=("arp@20", "mean"),
            popular_exp_pct=("popular_group_exposure_share_pct", "mean"),
            rare_exp_pct=("rare_group_exposure_share_pct", "mean"),
        )
        .reset_index()
    )
    out.insert(0, "dataset", dataset_label)
    return out


cross_df = pd.concat(
    [
        summarize_dataset(digi_acc, digi_pb, "Diginetica"),
        summarize_dataset(model_accuracy_df, popularity_bias_df, "Yoochoose 1/32"),
    ],
    ignore_index=True,
)
cross_df.to_csv(TABLES_DIR / "diginetica_vs_yoochoose_bias_comparison.csv", index=False)
display(cross_df)

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2))
ds_palette = {"Diginetica": "#4299e1", "Yoochoose 1/32": "#ed8936"}
m_order = ["GRU4REC", "NARM", "SASREC"]

for ax, col, title, ylabel, fmt in [
    (axes[0, 0], "recall20", "Recall@20 (Warm All-prefix)", "Recall@20", "%.3f"),
    (axes[0, 1], "coverage_warm_pct", "Warm Catalog Coverage@20 (%)", "Coverage (%)", "%.1f"),
    (axes[1, 0], "gini_warm", "Warm Recommendation Gini@20 (↓ ít bias hơn)", "Gini@20", "%.3f"),
    (axes[1, 1], "popular_exp_pct", "Popular Group Exposure Share (%)", "Exposure (%)", "%.1f"),
]:
    sns.barplot(data=cross_df, x="model", y=col, hue="dataset", order=m_order, palette=ds_palette, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, fontsize=8.5, padding=2)

plt.suptitle("So sánh chéo Accuracy & Popularity Bias: Diginetica vs. Yoochoose 1/32", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "diginetica_vs_yoochoose_bias_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Kết luận tổng hợp File 06 & Bàn giao sang File 07 (Hình học Embedding)

In [ ]:
acc_m = acc_seed_summary.set_index("model")
pb_m = pb_seed_summary.set_index("model")
best_acc_model = str(acc_m["recall@20_mean"].idxmax())
worst_gini_model = str(pb_m["gini_warm@20_mean"].idxmax())
best_gini_model = str(pb_m["gini_warm@20_mean"].idxmin())
best_cov_model = str(pb_m["coverage_warm@20_mean"].idxmax())

findings = [
    ("Accuracy tốt nhất (Recall/MRR/NDCG@20)", f"{best_acc_model} (Recall@20={acc_m.loc[best_acc_model, 'recall@20_mean']:.4f}, MRR@20={acc_m.loc[best_acc_model, 'mrr@20_mean']:.4f})", "Trùng khớp 100% thứ hạng với Diginetica"),
    ("Popularity Bias mạnh nhất (Gini/ARP/Pop Exp)", f"{worst_gini_model} (Gini={pb_m.loc[worst_gini_model, 'gini_warm@20_mean']:.4f}, ARP={pb_m.loc[worst_gini_model, 'arp@20_mean']:.1f}, Pop Exp={pb_m.loc[worst_gini_model, 'popular_group_exposure_share_pct_mean']:.2f}%)", "Khác nhẹ Diginetica (NARM cao nhất về Gini), nhưng khoảng cách GRU4Rec/NARM đều rõ hơn SASRec"),
    ("Popularity Bias thấp nhất & Coverage cao nhất", f"{best_gini_model} (Gini={pb_m.loc[best_gini_model, 'gini_warm@20_mean']:.4f}, Cov={pb_m.loc[best_cov_model, 'coverage_warm@20_mean']*100:.2f}%, Rare Exp={pb_m.loc[best_gini_model, 'rare_group_exposure_share_pct_mean']:.2f}%)", "SASRec đạt trade-off Accuracy–Diversity tốt nhất trên cả Diginetica và Yoochoose"),
    ("Khuếch đại Popular & bóp nghẹt Rare", "Popular (20.23% catalog, 83.78% train) nhận ~88–89% exposure; Rare (23.38% catalog, 2.00% train) chỉ nhận ~1.1–1.4%", "Mức độ bất bình đẳng đầu ra trên Yoochoose dữ dội hơn rõ rệt so với Diginetica (62–64% vs 4–5%)"),
    ("Bất lợi kép của nhóm Rare Target", f"Recall@20 nhóm Popular cao hơn nhóm Rare tới {strat_gap_df['recall_gap_pop_minus_rare_pp'].min():.1f}–{strat_gap_df['recall_gap_pop_minus_rare_pp'].max():.1f} điểm %", "Giống Diginetica: item hiếm vừa khó dự đoán đúng vừa nhận cực ít lượt hiển thị"),
    ("Hiệu ứng đặc thù Special Offer ('S')", "Nhóm item 'S' (~7.3% warm catalog) chiếm tỷ trọng lớn exposure Top-20 và có hệ số khuếch đại > 1", "Là nguồn cấu trúc chính đẩy Gini và ARP trên Yoochoose lên rất cao"),
    ("Độ ổn định qua 2 random seed (2024 & 2025)", "Thứ hạng SASRec > NARM > GRU4Rec theo cả Accuracy lẫn giảm Gini/ARP giữ nguyên ở cả 2 seed", "Kết luận không phải nhiễu ngẫu nhiên từ một lần khởi tạo"),
]

findings_df = pd.DataFrame(findings, columns=["khia_canh", "ket_qua_yoochoose", "doi_chieu_diginetica"])
findings_df.to_csv(TABLES_DIR / "file06_key_findings.csv", index=False)
display(findings_df)
print("\nĐã lưu toàn bộ bảng và biểu đồ File 06 vào:", YOO_PATHS["results_dir"])